<a href="https://colab.research.google.com/github/EmePin/Analisis-de-datos/blob/main/ACTAS_TALLER_B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.- descargar exceles

2.- montar drive

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ruta = "/content/drive/MyDrive/Colab Notebooks/actas/b1/taller/*.xlsx"

In [ ]:
import pandas as pd
import glob
import unicodedata
import os
import re
import numpy as np

In [ ]:
archivos = glob.glob(ruta)

In [ ]:
print("Archivos encontrados:")
print(archivos)

In [ ]:
# Lista para guardar todos los dataframes
lista_dfs = []

for archivo in archivos:

    # Leer cada archivo
    df = pd.read_excel(archivo)

    # Obtener solo el nombre del archivo (sin ruta)
    nombre = os.path.basename(archivo)

    # Buscar patrón tipo B1A, B1B, B1C, etc.
    match = re.search(r'B(\d)([A-Z])', nombre)

    if match:
        grado = match.group(1)
        grupo = match.group(2)
    else:
        grado = None
        grupo = None
   # Agregar columnas nuevas
    df["grado"] = grado
    df["grupo"] = grupo

    # Guardar dataframe en lista
    lista_dfs.append(df)

# Unir todos los dataframes
df_final = pd.concat(lista_dfs, ignore_index=True)

# Guardar archivo final
ruta_salida = "/content/drive/MyDrive/Colab Notebooks/actas/b1/cultura/ACTAS_UNIDAS.xlsx"
df_final.to_excel(ruta_salida, index=False)

print("Archivo final creado correctamente ✅")
print("Ruta:", ruta_salida)

In [ ]:
len(df_final)

In [ ]:
import unicodedata

def remove_accents(input_str):
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    return ''.join([c for c in nfkd_form if not unicodedata.combining(c)])

df_final.columns = df_final.columns.map(lambda x: remove_accents(x.lower().replace(' ', '_')))
print("Nuevos nombres de columnas (sin acentos):")
print(df_final.columns)
print("\nPrimeras 5 filas con los nuevos nombres de columnas:")
print(df_final.head())

In [ ]:
df_final = df_final.drop(columns=['lista', 'alumno'])
print("Columnas actualizadas después de la eliminación:")
print(df_final.columns)
print("\nPrimeras 5 filas con las columnas actualizadas:")
print(df_final.head())

In [ ]:
alumnos_por_grupo = df_final.groupby('grupo').size()
print("Número de alumnos por grupo:")
print(alumnos_por_grupo)

In [ ]:
promedio_grupal = df_final.groupby('grupo').mean(numeric_only=True)
print("Promedio grupal de las actividades:")
print(promedio_grupal)

In [ ]:
promedio_grupal_redondeado = promedio_grupal['sumatoria'].round(1)
print("Promedio de 'sumatoria' por grupo (redondeado a 1 decimal):")
print(promedio_grupal_redondeado)

In [ ]:
print("Promedio del examen teórico y práctico en Moodle por grupo:")
print(promedio_grupal['examen_teorico_y_practico_en_moodle'].round(1))

In [ ]:
promedio_examen_base_10 = (promedio_grupal['examen_teorico_y_practico_en_moodle'] / 5).round(1)
print("Promedio del examen teórico y práctico en Moodle (escala 0-10):")
print(promedio_examen_base_10)

In [ ]:
columnas_para_ev_continua = [col for col in df_final.columns if col not in ['examen_teorico_y_practico_en_moodle', 'sumatoria', 'grado', 'grupo']]
df_final['ev_continua'] = df_final[columnas_para_ev_continua].sum(axis=1)

print("Primeras 5 filas con todas las columnas:")
print(df_final.head())

In [ ]:
max_raw_ev_continua = 50 # Asumiendo 10 (kahoot) + 20 (practica) + 20 (investigacion) + 10 (punto_extra)

# La ev_continua ahora será escalada directamente a una base 0-10 para su visualización.
df_final['ev_continua_0_10'] = (df_final['ev_continua'] / max_raw_ev_continua) * 10

promedio_ev_continua_grupal_0_10 = df_final.groupby('grupo')['ev_continua_0_10'].mean().round(1)

print("Promedio de la evaluación continua por grupo (escala 0-10):")
print(promedio_ev_continua_grupal_0_10)

In [ ]:
# Escalar las calificaciones individuales del examen a una base de 0-10
df_final['examen_moodle_0_10'] = (df_final['examen_teorico_y_practico_en_moodle'] / 5).round(1)

# Filtrar los estudiantes que reprobaron el examen (calificación < 6 en escala 0-10)
reprobados_examen = df_final[df_final['examen_moodle_0_10'] < 6]

# Contar el número de reprobados por grupo
conteo_reprobados_por_grupo = reprobados_examen.groupby('grupo').size()

print("Número de estudiantes reprobados en el examen por grupo (calificación < 6 en escala 0-10):")
print(conteo_reprobados_por_grupo)

In [ ]:
conteo_aprobados_por_grupo = alumnos_por_grupo - conteo_reprobados_por_grupo

print("Número de estudiantes aprobados en el examen por grupo:")
print(conteo_aprobados_por_grupo)

In [ ]:
# Filtrar los estudiantes que reprobaron la evaluación continua (calificación < 6 en escala 0-10)
reprobados_ev_continua = df_final[df_final['ev_continua_0_10'] < 6]

# Contar el número de reprobados por grupo
conteo_reprobados_ev_continua_por_grupo = reprobados_ev_continua.groupby('grupo').size()

print("Número de estudiantes reprobados en evaluación continua por grupo (calificación < 6 en escala 0-10):")
print(conteo_reprobados_ev_continua_por_grupo)

In [ ]:
# Filtrar los estudiantes que aprobaron la evaluación continua (calificación >= 6 en escala 0-10)
aprobados_ev_continua = df_final[df_final['ev_continua_0_10'] >= 6]

# Contar el número de aprobados por grupo
conteo_aprobados_ev_continua_por_grupo = aprobados_ev_continua.groupby('grupo').size()

print("Número de estudiantes aprobados en evaluación continua por grupo:")
print(conteo_aprobados_ev_continua_por_grupo)

In [ ]:
# Calcular el promedio final de cada estudiante (50% evaluación continua + 50% examen)
df_final['promedio_final'] = (df_final['ev_continua_0_10'] * 0.5) + (df_final['examen_moodle_0_10'] * 0.5)

# Filtrar los estudiantes que reprobaron el promedio final (calificación < 6)
reprobados_promedio_final = df_final[df_final['promedio_final'] < 6]

# Contar el número de reprobados por grupo
conteo_reprobados_promedio_final_por_grupo = reprobados_promedio_final.groupby('grupo').size()

print("Número de estudiantes reprobados por promedio final por grupo (calificación < 6):")
print(conteo_reprobados_promedio_final_por_grupo)

In [ ]:
# Filtrar los estudiantes que aprobaron el promedio final (calificación >= 6)
aprobados_promedio_final = df_final[df_final['promedio_final'] >= 6]

# Contar el número de aprobados por grupo
conteo_aprobados_promedio_final_por_grupo = aprobados_promedio_final.groupby('grupo').size()

print("Número de estudiantes aprobados por promedio final por grupo:")
print(conteo_aprobados_promedio_final_por_grupo)